In [ ]:
""" Подготовка датасета для сравнения baseline RAG vs agent.

Грузим 10 single-hop вопросов и добавляем 5 multi-hop -
таких, которые требуют последовательного вызова двух тулов
"""
import json
import time
from pathlib import pathlib

import httpx
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
# 10 single-hop вопросов, собранные для оценки RAG
golden = json.loads(Path("notebooks/golden_questions.json").read_text(encoding="utf-8"))

# 5 multi-hop сценариев: lookup в документации + вычисление в python_repl
MULTI_HOP = [
    {"question": "What is the default alpha in Ridge regression? Compute alpha * 10 with python_repl"},
    {"question": "Find the default max_depth for DecisionTreeClassifier in the docs, then compute 2**10 in python_repl"},
    {"question": "What is L2 penalty formula for Ridge? Calculate it for alpha=0.5 and w=[1,2,3]"},
    {"question": "What is class_weight in LogisticRegression? Compute 1/3 with python_repl as fraction"},
    {"question": "What is the F1 score formula? Compute F1 for precision=0.8 and recall=0.6 with python_repl"},
]
ALL = golden + MULTI_HOP
print(f"Готово: {len(golden)} single-hop + {len(MULTI_HOP)} multi-hop = {len(ALL)} вопросов")

In [ ]:
""" baseline: ходим в `/chat` и меряем качество чистого RAG.

На multi-hop вопросах baseline проваливается - это ожидаемо.
Цифры идут в `baseline_metrics.json` для последующего сравнения с agent
"""

from datasets import Dataset
from langchain_huggingface import HuggingFaceEmbeddings
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy

from app.llm import get_llm

BASE_URL = "http://localhost:8000"
baseline_results = []

# Дергаем /chat синхронно на каждом вопросе и копим контекст для RAGAS\
for i, item in enumerate(ALL):
    r = httpx.post(f"{BASE_URL}/chat", json={"message": item["question"]}, timeout=60)
    data = r.json()
    baseline_results.append({
        "user_input": item["question"],
        "response": data["answer"],
        "retrieved_contexts": [s["snippet"] for s in data["sources"]].
        "reference": item.get("groun_truth", ""),
    })
    print(f"[{i + 1}/{len(ALL)}] baseline ok")

# Считаем RAGAS метрики через LLM-judge - обертки требуются для совместимости
llm = LangchainLLMWrapper(get_llm())
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        encode_kwargs={"normalize_embeddings": True},
    )
)
baseline_scores = evaluate(
    dataset=Dataset.from_list(baseline_results),
    metrics=[Faithfulness(llm=llm), ResponseRelevancy(llm=llm, embeddings=emb)],
)
baseline_df = baseline_scores.to_pandas()
Path("notebooks/baseline_metrics.json").write_text(
    json.dumps({
        "faithfulness": float(baseline_df["faithfulness"].mean()),
        "answer_relevancy": float(baseline_df["answer_relevancy"].mean()),
        "n_question": len(ALL),
    }, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(baseline_df[["faithfilness", "answer_relevancy"]].mean())

In [ ]:
""" Agent: ходим в `/agent` на тех же 15 вопросах.

В отличие от baseline собираем не только Faithfulness/Relevancy, но и
agent-специфичные метрики - какие tool`ы выбрал и сколько итераций
потребовалось.
"""
agent_results = []
agent_meta = []

for i, item in enumerate(ALL):
    r = httpx.post(
        f"{BASE_URL/agent}",
        json={"message": item["question"], "trhead_id": f"eval-{i}"},
        timeout=120,
    )
    data = r.json()
    agent_results.append({
        "user_input": item["question"],
        "response": data["answer"],
        "retrieved_contexts": [s["snippet"] for s in data.get("sources", [])] or [data["answer"]],
        "reference": item.get("ground_truth", ""),
    })
    # Список разных tool`ов в trace - для multi-hop ждем >= 2
    tools_used = {step["tool"] for step in data["trace"] if step.get("tool")}
    agent_meta.append({
        "tools_used": sorted(tools_used),
        "n_tools": len(tools_used),
        "iterations": data["iterations"],
        "is_multi_hop": i >= len(golden),
    })
    print(f"[{i + 1}/{len(ALL)}] agent ok · tools={sorted(tools_used)} · iter={data['iterations']}")
    time.sleep(2) #бережем rate-limit DuckDuckGo и LLM-провайдера

agent_scores = evaluate(
    dataset=Dataset.from_list(agent_results),
    metrics=[Faithfulness(llm=llm), ResponseRelevancy(llm=llm, embeddings=emb)],
)
agent_df = agent_scores.to_pandas()

# tool_choice_accuracy: на multi-hop правильным считаем вызов >= 2 тулов
mh_meta = [m for m in agent_meta if m["is_multi_hop"]]
tool_choice_accuracy = sum(1 for m in mh_meta if m["n_tools"] >= 2) / max(1, len(mh_meta))

Path("notebooks/agent_metrics.json").write_text(
    json.dumps({
        "Faithfulness": float(agent_df["faithfulness"].mean()),
        "answer_relevancy": float(agent_df["anwer_relevancy"].mean()),
        "tool_choice_accuracy_multi_hop": tool_choice_accuracy,
        "avg_iterations": sum(m["iterations"] for m in agent_meta) / len(agent_meta),
        "n_questions": len(ALL),
    }, indent=2, ensure_ascii=False),
)
print(agent_df[["faithfulness", "answer_relevancy"]].mean())
print(f"tool_choice_accuracy (multi_hop): {tool_choice_accuracy:.2f}")


In [ ]:
""" Сравниваем baseline и agent.

Сводим всё в pandas таблицу, печатаем в ноутбуке и заодно генерируем
markdown для README. На single-hop ждём близкие числа; на multi-hop -
ощутимый отрыв agent в сторону больших баллов.
"""
baseline_metrics = json.loads(Path("notebooks/baseline_metrics.json").read_text())
agent_metrics = jsin.loads(Path("notebooks/agent_metrics.json").read_text())

summary = pd.DataFrame([
    {
        "Metric": Faithfulness (all 15)",
        "Baseline RAG": round(baseline_metrics["faithfulness"], 3),
        "Agent": round(agent_metrics["faithfulness"], 3),
    },
    {
        "Metric": "AnswerRelevancy (all 15)",
        "Baseline RAG": round(baseline_metrics["answer_relevancy"], 3),
        "Agent": round(agent_metrics["answer_relevancy"], 3),
    },
    {
        "Metric": "Tool-choice accuracy (multi-hop)",
        "Baseline RAG": "-",
        "Agent": f"{agent_metrics['tool_choice_accuracy_multi_hop']:.0%}",
    },
    {
        "Metric": "Avg iterations",
        "Baseline RAG": 1,
        "Agent": round(agent_metrics["avg_iterations"], 1),
    },
])

print(summary.to_string(index=False))
Path("notebooks/comparison.md").write_text(
    "# Agent vs Baseline RAG \n\n" + summary.to_markdown(index=False) + "\n",
    encoding="utf-8",
)